# Module 5.3 — 4-bit Quantisation: GPTQ, AWQ, GGUF

**Goal:** Quarter VRAM vs fp16. Produce three int4 artifacts from the fine-tuned DeskMate decoder: a GPTQ GPU model, an AWQ GPU model, and a GGUF file for CPU inference. Benchmark all four (including fp16 baseline) on VRAM, throughput, and ROUGE-L.

## 1. Install Dependencies

In [ ]:
# Run once in Colab — each library installs its own CUDA kernels
# !pip install -q auto-gptq optimum
# !pip install -q autoawq
# !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
# !pip install -q rouge-score transformers accelerate

import os, time, json, subprocess
from pathlib import Path
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Configuration

In [ ]:
BASE_MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'
# After Module 3.4 merge, use merged adapter path:
# BASE_MODEL = 'models/deskmate-merged/'

GPTQ_DIR = 'models/deskmate-gptq-4bit/'
AWQ_DIR  = 'models/deskmate-awq-4bit/'
GGUF_FP16 = 'models/deskmate-f16.gguf'
GGUF_Q4   = 'models/deskmate-Q4_K_M.gguf'
GGUF_Q8   = 'models/deskmate-Q8_0.gguf'

SMOKE_TEST = True   # set False to run full quantisation + eval
MAX_NEW_TOKENS = 100
N_BENCH_RUNS   = 5

# Calibration examples: 10 representative DeskMate tickets
CALIB_TEXTS = [
    'I was charged twice for my subscription last month.',
    'How do I reset my 2FA device?',
    'The CSV export button is not working.',
    'Can I get a refund after 45 days?',
    'My iOS app keeps crashing after the latest update.',
    'How do I enable SSO for my organisation?',
    'What is the uptime SLA on the Enterprise plan?',
    'I cannot log in after changing my email address.',
    'Does DeskMate integrate with Slack?',
    'How do I export all my data before cancelling?',
]

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)
print('Config OK')

## 3. Gold Evaluation Set

In [ ]:
GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'ref': 'Contact support with your invoice numbers. We investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my 2FA device?',
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'ref': 'This is a known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
    {'ticket': 'Can I get a refund after 30 days?',
     'ref': 'DeskMate offers a 30-day money-back guarantee. Refunds after 30 days are case by case.'},
    {'ticket': 'My iOS app keeps crashing on iPhone 14.',
     'ref': 'A crash affecting iOS 17 was fixed in app version 2.1.3. Please update from the App Store.'},
]

print(f'Gold examples: {len(GOLD)}')

## 4. ROUGE-L Helper

In [ ]:
from rouge_score import rouge_scorer as rs_module
import numpy as np

_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_l(pred, ref):
    return round(_scorer.score(ref, pred)['rougeL'].fmeasure, 4)

def vram_gb():
    if not torch.cuda.is_available(): return 0.0
    return round(torch.cuda.memory_allocated() / 1e9, 2)

## 5. fp16 Baseline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating fp16 baseline.')
    print('[Simulated] fp16 VRAM: 3.0 GB  |  tokens/sec: 28.0  |  ROUGE-L: 0.47')
    fp16_stats = {'vram': 3.0, 'tps': 28.0, 'rouge': 0.47, 'size_gb': 3.0}
    tokenizer = None
    model_fp16 = None
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    torch.cuda.reset_peak_memory_stats()
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    fp16_vram = vram_gb()
    print(f'fp16 VRAM: {fp16_vram} GB')
    fp16_stats = {'vram': fp16_vram, 'tps': None, 'rouge': None, 'size_gb': 3.0}

## 6. Throughput & ROUGE-L Helpers

In [ ]:
def bench_hf(model, tok, n_runs=N_BENCH_RUNS):
    if model is None: return None
    prompt = 'I need help with my billing.'
    inputs = tok(prompt, return_tensors='pt').to(DEVICE)
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return round(MAX_NEW_TOKENS / float(np.median(times)), 1)

def eval_rouge_hf(model, tok):
    if model is None: return None
    scores = []
    for ex in GOLD:
        msgs = [
            {'role': 'system', 'content': 'You are DeskMate, a concise support assistant.'},
            {'role': 'user',   'content': f"Ticket: {ex['ticket']}"},
        ]
        if hasattr(tok, 'apply_chat_template'):
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"Ticket: {ex['ticket']}\nAnswer:"
        inputs = tok(prompt, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        pred = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        scores.append(rouge_l(pred, ex['ref']))
    return round(sum(scores)/len(scores), 4)

## 7. GPTQ 4-bit Quantisation

In [ ]:
# GPTQ uses second-order weight correction (Hessian compensation) to minimise
# layer output error — not just weight approximation error.

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating GPTQ quantisation.')
    print('[Simulated] GPTQ VRAM: 0.75 GB  |  tokens/sec: 35.0  |  ROUGE-L: 0.455')
    gptq_stats = {'vram': 0.75, 'tps': 35.0, 'rouge': 0.455, 'size_gb': 0.95}
else:
    from transformers import GPTQConfig

    # Free fp16 model first
    if model_fp16 is not None:
        del model_fp16; torch.cuda.empty_cache()

    gptq_config = GPTQConfig(
        bits=4,
        dataset=CALIB_TEXTS,
        tokenizer=tokenizer,
        group_size=128,
    )
    torch.cuda.reset_peak_memory_stats()
    model_gptq = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=gptq_config,
        device_map='auto',
        token=HF_TOKEN,
    )
    gptq_vram = vram_gb()
    print(f'GPTQ VRAM: {gptq_vram} GB')
    model_gptq.save_pretrained(GPTQ_DIR)
    tokenizer.save_pretrained(GPTQ_DIR)
    gptq_tps   = bench_hf(model_gptq, tokenizer)
    gptq_rouge = eval_rouge_hf(model_gptq, tokenizer)
    gptq_size  = sum(f.stat().st_size for f in Path(GPTQ_DIR).rglob('*.safetensors')) / 1e9
    gptq_stats = {'vram': gptq_vram, 'tps': gptq_tps, 'rouge': gptq_rouge, 'size_gb': round(gptq_size, 2)}
    del model_gptq; torch.cuda.empty_cache()

print('GPTQ stats:', gptq_stats)

## 8. AWQ 4-bit Quantisation

In [ ]:
# AWQ scales salient weight channels UP before quantising them,
# so the 4-bit bins cover the important range more precisely.
# At inference, activation inputs are scaled DOWN by the same factor.

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating AWQ quantisation.')
    print('[Simulated] AWQ VRAM: 0.75 GB  |  tokens/sec: 38.0  |  ROUGE-L: 0.462')
    awq_stats = {'vram': 0.75, 'tps': 38.0, 'rouge': 0.462, 'size_gb': 0.90}
else:
    from awq import AutoAWQForCausalLM

    model_awq = AutoAWQForCausalLM.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    tok_awq   = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

    quant_config = {
        'zero_point': True,
        'q_group_size': 128,
        'w_bit': 4,
        'version': 'GEMM',
    }
    model_awq.quantize(tok_awq, quant_config=quant_config, calib_data=CALIB_TEXTS)
    model_awq.save_quantized(AWQ_DIR)
    tok_awq.save_pretrained(AWQ_DIR)

    torch.cuda.reset_peak_memory_stats()
    model_awq_inf = AutoAWQForCausalLM.from_quantized(AWQ_DIR, fuse_layers=True)
    awq_vram  = vram_gb()
    awq_tps   = bench_hf(model_awq_inf, tok_awq)
    awq_rouge = eval_rouge_hf(model_awq_inf, tok_awq)
    awq_size  = sum(f.stat().st_size for f in Path(AWQ_DIR).rglob('*.safetensors')) / 1e9
    awq_stats = {'vram': awq_vram, 'tps': awq_tps, 'rouge': awq_rouge, 'size_gb': round(awq_size, 2)}
    del model_awq, model_awq_inf; torch.cuda.empty_cache()

print('AWQ stats:', awq_stats)

## 9. GGUF Export (llama.cpp)

GGUF export requires the merged fp16 HF model on disk and the `llama.cpp` tools. In a real run, execute the shell commands below. In smoke-test mode we simulate the output.

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating GGUF export.')
    print('[Simulated] Q4_K_M size: 0.95 GB  |  CPU tokens/sec: 8.5  |  ROUGE-L: 0.449')
    print('[Simulated] Q8_0    size: 1.60 GB  |  CPU tokens/sec: 5.2  |  ROUGE-L: 0.464')
    gguf_q4_stats = {'vram': 0.0, 'ram_gb': 0.95, 'tps': 8.5,  'rouge': 0.449, 'size_gb': 0.95}
    gguf_q8_stats = {'vram': 0.0, 'ram_gb': 1.60, 'tps': 5.2,  'rouge': 0.464, 'size_gb': 1.60}
else:
    # Step 1: Clone llama.cpp and install Python deps
    # !git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
    # !pip install -q -r /tmp/llama.cpp/requirements.txt

    # Step 2: Convert HF model → GGUF fp16
    # !python /tmp/llama.cpp/convert_hf_to_gguf.py {BASE_MODEL} \
    #         --outtype f16 --outfile {GGUF_FP16}

    # Step 3: Quantise to Q4_K_M and Q8_0
    # !/tmp/llama.cpp/quantize {GGUF_FP16} {GGUF_Q4} Q4_K_M
    # !/tmp/llama.cpp/quantize {GGUF_FP16} {GGUF_Q8} Q8_0

    # Step 4: File sizes
    q4_size = Path(GGUF_Q4).stat().st_size / 1e9 if Path(GGUF_Q4).exists() else 0.0
    q8_size = Path(GGUF_Q8).stat().st_size / 1e9 if Path(GGUF_Q8).exists() else 0.0
    print(f'Q4_K_M: {q4_size:.2f} GB   Q8_0: {q8_size:.2f} GB')
    gguf_q4_stats = {'vram': 0.0, 'ram_gb': q4_size, 'tps': None, 'rouge': None, 'size_gb': round(q4_size, 2)}
    gguf_q8_stats = {'vram': 0.0, 'ram_gb': q8_size, 'tps': None, 'rouge': None, 'size_gb': round(q8_size, 2)}

## 10. GGUF CPU Inference (llama-cpp-python)

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — llama-cpp-python inference skipped.')
    print('Simulated reply: [GGUF Q4_K_M response to billing query]')
else:
    from llama_cpp import Llama

    llm_q4 = Llama(
        model_path=GGUF_Q4,
        n_ctx=2048,
        n_threads=os.cpu_count(),
        verbose=False,
    )

    def gguf_chat(llm, ticket):
        resp = llm.create_chat_completion(
            messages=[
                {'role': 'system', 'content': 'You are DeskMate, a concise support assistant.'},
                {'role': 'user',   'content': f'Ticket: {ticket}'},
            ],
            max_tokens=MAX_NEW_TOKENS,
        )
        return resp['choices'][0]['message']['content'].strip()

    # Benchmark throughput (tokens/sec on CPU)
    times = []
    for _ in range(N_BENCH_RUNS):
        t0 = time.perf_counter()
        gguf_chat(llm_q4, GOLD[0]['ticket'])
        times.append(time.perf_counter() - t0)
    gguf_q4_tps = round(MAX_NEW_TOKENS / float(np.median(times)), 1)
    gguf_q4_rouge = round(sum(rouge_l(gguf_chat(llm_q4, ex['ticket']), ex['ref']) for ex in GOLD) / len(GOLD), 4)
    gguf_q4_stats['tps']   = gguf_q4_tps
    gguf_q4_stats['rouge'] = gguf_q4_rouge
    print(f'GGUF Q4_K_M  tokens/sec (CPU): {gguf_q4_tps}')
    print(f'GGUF Q4_K_M  ROUGE-L:          {gguf_q4_rouge}')

## 11. Comparison Chart

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

formats = ['fp16', 'GPTQ 4b', 'AWQ 4b', 'GGUF Q4', 'GGUF Q8']
sizes   = [fp16_stats['size_gb'], gptq_stats['size_gb'],
           awq_stats['size_gb'],  gguf_q4_stats['size_gb'], gguf_q8_stats['size_gb']]
tps_vals = [fp16_stats['tps'], gptq_stats['tps'],
            awq_stats['tps'],  gguf_q4_stats['tps'], gguf_q8_stats['tps']]
rouges   = [fp16_stats['rouge'], gptq_stats['rouge'],
            awq_stats['rouge'],  gguf_q4_stats['rouge'], gguf_q8_stats['rouge']]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', 'coral', 'seagreen', 'orange', 'mediumpurple']

# File size
ax = axes[0]
ax.bar(formats, sizes, color=colors)
ax.set_ylabel('Size (GB)')
ax.set_title('Model File Size')
ax.tick_params(axis='x', rotation=20)

# Throughput
ax = axes[1]
tps_clean = [v if v is not None else 0 for v in tps_vals]
ax.bar(formats, tps_clean, color=colors)
ax.set_ylabel('Tokens / second')
ax.set_title('Throughput (GPU unless GGUF)')
ax.tick_params(axis='x', rotation=20)

# ROUGE-L
ax = axes[2]
rouge_clean = [v if v is not None else 0 for v in rouges]
ax.bar(formats, rouge_clean, color=colors)
ax.set_ylim(0.3, 0.55)
ax.set_ylabel('ROUGE-L')
ax.set_title('Quality (ROUGE-L)')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('quant_4bit_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: quant_4bit_comparison.png')

## 12. Summary Table

In [ ]:
df = pd.DataFrame({
    'Format':     formats,
    'Size (GB)':  sizes,
    'Tokens/sec': tps_vals,
    'ROUGE-L':    rouges,
    'Runs on':    ['GPU', 'GPU', 'GPU', 'CPU', 'CPU'],
})
print(df.to_string(index=False))

## 13. Checkpoint: GPTQ vs GGUF

In [ ]:
checkpoint = '''
Checkpoint: GPTQ vs GGUF — when do you reach for each?

GPTQ: Reach for it when deploying on a GPU server (vLLM, FastAPI).
  - Output: HuggingFace-compatible directory, loadable with transformers.
  - Faster GPU inference via exllama/triton int4 kernels.
  - Requires GPU at inference time.

GGUF: Reach for it when deploying on CPU, laptops, or removing the PyTorch dependency.
  - Output: a single portable .gguf file, run by llama.cpp or Ollama.
  - No GPU required; memory-mapped loading.
  - Best for local/offline/edge deployment (Module 6.4).

Key distinction: GPTQ = better int4 GPU inference.
                 GGUF = CPU-first portability.
'''
print(checkpoint)

## 14. Save Report

In [ ]:
report = [
    '# 4-bit Quantisation Report\n',
    f'Model: {BASE_MODEL}',
    f'Smoke test: {SMOKE_TEST}\n',
    '| Format | Size (GB) | Tokens/sec | ROUGE-L | Runs on |',
    '|--------|-----------|------------|---------|---------|',
]
for fmt, sz, tps, rl, hw in zip(
        formats, sizes, tps_vals, rouges, ['GPU','GPU','GPU','CPU','CPU']):
    report.append(f'| {fmt} | {sz} | {tps} | {rl} | {hw} |')

report += [
    '',
    '## Checkpoint',
    '',
    '**GPTQ** — use for GPU serving (vLLM/FastAPI); HF-compatible output.',
    '**GGUF** — use for CPU/local/offline deployment (llama.cpp/Ollama).',
]

with open('reports/quant_4bit_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/quant_4bit_report.md')
print('\n=== Module 5.3 Complete ===')